# YOLO ROI Crop For Requested Folders

Notebook does the following:
1. Runs `tools/yolo_crop_roi.py` with model `runs/detect/runs/petri_curcle/yolo26n_petri_curcle/weights/best.pt`.
2. Applies ROI crop to images from:
   - `Новая папка/images/train`
   - `Новая папка (2)/рч/img`
3. Applies the same crop to masks from:
   - `Новая папка (2)/рч/masks_human`
   - `Новая папка (2)/рч/masks_instances`
   - `Новая папка (2)/рч/masks_machine`
4. Recalculates YOLO-seg labels from `Новая папка/labels/train` for cropped images and saves them to `runs/yolo_roi_new_folders_notebook/labels/train`.

In [ ]:
from pathlib import Path
import csv
import json
import re
import subprocess
import sys

import cv2
import numpy as np

PROJECT_ROOT = Path.cwd()
MODEL_PATH = PROJECT_ROOT / "runs/detect/runs/petri_curcle/yolo26n_petri_curcle/weights/best.pt"

IMAGE_DIRS = [
    PROJECT_ROOT / "Новая папка (2)/рч/img",
    PROJECT_ROOT / "Новая папка/images/train",
]

MASK_DIRS = [
    PROJECT_ROOT / "Новая папка (2)/рч/masks_human",
    PROJECT_ROOT / "Новая папка (2)/рч/masks_instances",
    PROJECT_ROOT / "Новая папка (2)/рч/masks_machine",
]

LABEL_DIR = PROJECT_ROOT / "Новая папка/labels/train"

OUT_DIR = PROJECT_ROOT / "runs/yolo_roi_new_folders_notebook"
OUT_LABEL_DIR = OUT_DIR / "labels/train"

CONF = 0.05
IOU = 0.5
IMGSZ = 1600
PAD_FRAC = 0.03
FALLBACK_FULL_IMAGE = True
SAVE_OVERLAY = True

print("Project root:", PROJECT_ROOT)
print("Model:", MODEL_PATH)
print("Out dir:", OUT_DIR)

In [ ]:
assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"
for p in IMAGE_DIRS + MASK_DIRS + [LABEL_DIR]:
    assert p.exists(), f"Path not found: {p}"

cmd = [
    sys.executable,
    "tools/yolo_crop_roi.py",
    "--model", str(MODEL_PATH),
    "--image_dirs", *[str(p) for p in IMAGE_DIRS],
    "--mask_dirs", *[str(p) for p in MASK_DIRS],
    "--out_dir", str(OUT_DIR),
    "--conf", str(CONF),
    "--iou", str(IOU),
    "--imgsz", str(IMGSZ),
    "--pad_frac", str(PAD_FRAC),
]

if FALLBACK_FULL_IMAGE:
    cmd.append("--fallback_full_image")
if SAVE_OVERLAY:
    cmd.append("--save_overlay")

print(" ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

In [ ]:
summary_path = OUT_DIR / "summary.json"
report_path = OUT_DIR / "crop_report.csv"

summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Report:", report_path)

In [ ]:
IMG_KEY_RE = re.compile(r"(IMG_\d+)", flags=re.IGNORECASE)


def extract_img_key(text: str) -> str:
    match = IMG_KEY_RE.search(text)
    if match:
        return match.group(1).upper()
    return Path(text).stem.upper()


def imread_shape(path: Path) -> tuple[int, int]:
    buffer = np.fromfile(str(path), dtype=np.uint8)
    image = cv2.imdecode(buffer, cv2.IMREAD_UNCHANGED)
    if image is None:
        raise ValueError(f"Failed to read image: {path}")
    h, w = image.shape[:2]
    return w, h


rows_by_key = {}
with (OUT_DIR / "crop_report.csv").open("r", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        rows_by_key[row["key"]] = row

OUT_LABEL_DIR.mkdir(parents=True, exist_ok=True)

written = 0
skipped = 0
for src_label in sorted(LABEL_DIR.glob("*.txt")):
    key = extract_img_key(src_label.stem)
    row = rows_by_key.get(key)
    if row is None or row["status"] not in {"cropped", "fallback_full_image"}:
        skipped += 1
        continue

    src_image = Path(row["source_image"])
    x1 = int(row["x1"])
    y1 = int(row["y1"])
    x2 = int(row["x2"])
    y2 = int(row["y2"])

    img_w, img_h = imread_shape(src_image)
    crop_w = max(x2 - x1, 1)
    crop_h = max(y2 - y1, 1)

    out_lines = []
    for raw_line in src_label.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) < 3:
            continue

        cls_id = parts[0]
        coords = [float(v) for v in parts[1:]]
        if len(coords) % 2 != 0:
            continue

        new_coords = []
        inside_points = 0
        for xn, yn in zip(coords[0::2], coords[1::2]):
            x_abs = xn * img_w
            y_abs = yn * img_h

            x_new = (x_abs - x1) / crop_w
            y_new = (y_abs - y1) / crop_h

            if 0.0 <= x_new <= 1.0 and 0.0 <= y_new <= 1.0:
                inside_points += 1

            x_new = min(max(x_new, 0.0), 1.0)
            y_new = min(max(y_new, 0.0), 1.0)
            new_coords.extend([x_new, y_new])

        if inside_points == 0:
            continue

        out_line = " ".join([cls_id] + [f"{v:.6f}" for v in new_coords])
        out_lines.append(out_line)

    dst_label = OUT_LABEL_DIR / src_label.name
    dst_label.write_text("\n".join(out_lines) + ("\n" if out_lines else ""), encoding="utf-8")
    written += 1

print(f"Converted labels: {written}")
print(f"Skipped labels: {skipped}")
print("Output labels dir:", OUT_LABEL_DIR)

In [ ]:
all_out_labels = sorted(OUT_LABEL_DIR.glob("*.txt"))
print(f"Output label files: {len(all_out_labels)}")

for path in all_out_labels[:3]:
    print(f"\n{path.name}")
    for line in path.read_text(encoding="utf-8").splitlines()[:2]:
        print(line)